In [1]:
pip install ultralytics opencv-python numpy pyttsx3 speechrecognition


Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Stop Detection
import cv2
import pyttsx3
import speech_recognition as sr
import threading
from ultralytics import YOLO

# Load YOLOv8 model
model = YOLO("yolov8n.pt")  # Use yolov8s.pt for better accuracy

# Object width reference (in cm) for distance estimation
KNOWN_WIDTHS = {
    "person": 45, "bottle": 7, "chair": 50, "mobile phone": 7,
    "car": 180, "bus": 250, "truck": 250
}

FOCAL_LENGTH = 700  # Approximate focal length

# Initialize text-to-speech engine
engine = pyttsx3.init()

# Initialize speech recognition
recognizer = sr.Recognizer()
mic = sr.Microphone()

# Global variables
detection_active = True  # Start with detection enabled
frame = None
lock = threading.Lock()  # Thread synchronization

# Function to estimate distance
def estimate_distance(known_width, perceived_width):
    return (known_width * FOCAL_LENGTH) / perceived_width if perceived_width > 0 else None

# Function to format distance for speech
def format_distance(distance):
    if distance < 100:  # Less than 1m → use cm
        return f"{int(distance)} centimeters"
    else:  # 1m or more → use meters
        return f"{distance / 100:.1f} meters"

# Function to speak detected objects
def speak_objects(items):
    if items:
        message = ", ".join(items)
        engine.say(message)
        engine.runAndWait()

# Function to detect objects
def detect_objects():
    global frame, detection_active

    cap = cv2.VideoCapture(0)
    
    while cap.isOpened():
        ret, new_frame = cap.read()
        if not ret:
            break
        
        with lock:
            frame = new_frame.copy()

        if detection_active:
            results = model.predict(frame, verbose=False)
            detections = results[0].boxes
            detected_items = []

            for box in detections:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                x_min, y_min, x_max, y_max = map(int, box.xyxy[0])
                box_width = x_max - x_min

                if label in KNOWN_WIDTHS:
                    distance = estimate_distance(KNOWN_WIDTHS[label], box_width)
                    if distance:
                        formatted_distance = format_distance(distance)
                        detected_items.append(f"{label} is {formatted_distance} away")

                        # Draw bounding box
                        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                        cv2.putText(frame, f"{label}: {formatted_distance}", 
                                    (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                                    0.5, (0, 255, 0), 2)

            # Speak detected objects
            if detected_items:
                threading.Thread(target=speak_objects, args=(detected_items,), daemon=True).start()

        # Show the frame
        cv2.imshow("Object Detection", frame)

        # Press 'q' to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Function to listen for stop command
def listen_for_command():
    global detection_active
    while True:
        with mic as source:
            recognizer.adjust_for_ambient_noise(source)
            try:
                print("Listening for stop command...")
                audio = recognizer.listen(source, timeout=1)
                command = recognizer.recognize_google(audio).lower()
                print(f"Command recognized: {command}")

                if "stop detection" in command:
                    detection_active = False
                    engine.say("Detection stopped.")
                    engine.runAndWait()
            except sr.UnknownValueError:
                continue
            except sr.RequestError:
                print("Speech recognition service unavailable.")

# Main function
def main():
    # Start voice command listener thread
    listener_thread = threading.Thread(target=listen_for_command, daemon=True)
    listener_thread.start()

    # Start object detection
    detect_objects()

if __name__ == "__main__": 
    main()


In [ ]:
import cv2
import pyttsx3
import speech_recognition as sr
import threading
from ultralytics import YOLO

# Load YOLO model
model = YOLO("yolov8n.pt") 

# Object width reference (cm) for distance estimation
KNOWN_WIDTHS = {
    "person": 45, "bottle": 7, "chair": 50, "mobile phone": 7,
    "car": 180, "bus": 250, "truck": 250
}

FOCAL_LENGTH = 700  # Approximate focal length

# Initialize text-to-speech engine
engine = pyttsx3.init()

# Initialize speech recognition
recognizer = sr.Recognizer()
mic = sr.Microphone()

# Global variables
detection_active = False
frame = None
lock = threading.Lock()

# Function to estimate distance
def estimate_distance(known_width, perceived_width):
    return (known_width * FOCAL_LENGTH) / perceived_width if perceived_width > 0 else None

# Function to format distance
def format_distance(distance):
    return f"{int(distance)} centimeters" if distance < 100 else f"{distance / 100:.1f} meters"

# Function to speak detected objects
def speak_objects(items):
    if items:
        engine.say(", ".join(items))
        engine.runAndWait()

# Function to detect objects
def detect_objects():
    global frame, detection_active

    cap = cv2.VideoCapture(0)

    while cap.isOpened():
        ret, new_frame = cap.read()
        if not ret:
            break

        with lock:
            frame = new_frame.copy()

        if detection_active:
            results = model.predict(frame, verbose=False)
            detections = results[0].boxes
            detected_items = []

            for box in detections:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                x_min, y_min, x_max, y_max = map(int, box.xyxy[0])
                box_width = x_max - x_min

                if label in KNOWN_WIDTHS:
                    distance = estimate_distance(KNOWN_WIDTHS[label], box_width)
                    if distance:
                        formatted_distance = format_distance(distance)
                        detected_items.append(f"{label} is {formatted_distance} away")

                        # Draw bounding box
                        cv2.rectangle(frame, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                        cv2.putText(frame, f"{label}: {formatted_distance}", 
                                    (x_min, y_min - 10), cv2.FONT_HERSHEY_SIMPLEX, 
                                    0.5, (0, 255, 0), 2)

            # Speak detected objects
            if detected_items:
                threading.Thread(target=speak_objects, args=(detected_items,), daemon=True).start()

        # Show the frame
        cv2.imshow("Object Detection", frame)

        # Press 'q' to exit manually
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Function to listen for start/stop commands
def listen_for_commands():
    global detection_active
    while True:
        with mic as source:
            recognizer.adjust_for_ambient_noise(source, duration=0.3)  # Faster noise adjustment
            try:
                print("Listening for commands...")
                audio = recognizer.listen(source, timeout=0.5)  # Faster response
                command = recognizer.recognize_google(audio).lower()
                print(f"Command recognized: {command}")

                if "start detection" in command and not detection_active:
                    detection_active = True
                    engine.say("Starting object detection.")
                    engine.runAndWait()

                elif "stop detection" in command and detection_active:
                    detection_active = False
                    engine.say("Stopping detection.")
                    engine.runAndWait()

            except sr.UnknownValueError:
                continue  # Ignore unrecognized audio
            except sr.RequestError:
                print("Speech recognition service unavailable.")

# Main function
def main():
    # Start voice command listener thread
    listener_thread = threading.Thread(target=listen_for_commands, daemon=True)
    listener_thread.start()

    # Start object detection
    detect_objects()

if __name__ == "__main__":
    main()


In [ ]:
#Start Detection
import cv2
import pyttsx3
import speech_recognition as sr
import threading
import time
from ultralytics import YOLO

# Load optimized YOLOv8 model
model = YOLO("yolov8n.pt")  # Try yolov8s.pt for better speed-accuracy balance

# Object width reference (in cm) for distance estimation
KNOWN_WIDTHS = {
    "person": 45, "bottle": 7, "chair": 50, "mobile phone": 7,
    "car": 180, "bus": 250, "truck": 250
}

FOCAL_LENGTH = 700  # Approximate focal length

# Initialize text-to-speech engine
engine = pyttsx3.init()

# Initialize speech recognition
recognizer = sr.Recognizer()
mic = sr.Microphone()

# Global variables
detection_active = False
frame = None  # Latest frame (shared between threads)
lock = threading.Lock()  # Prevent race conditions

# Function to estimate distance
def estimate_distance(known_width, perceived_width):
    return (known_width * FOCAL_LENGTH) / perceived_width if perceived_width > 0 else None

# Function to process and detect objects in frames
def detect_objects():
    global frame, detection_active

    while True:
        if detection_active and frame is not None:
            with lock:
                frame_copy = frame.copy()  # Work on a copy to avoid race conditions

            results = model.predict(frame_copy, verbose=False)  # Optimized inference
            detections = results[0].boxes
            detected_items = []

            for box in detections:
                class_id = int(box.cls[0])
                label = model.names[class_id]
                x_min, y_min, x_max, y_max = map(int, box.xyxy[0])
                box_width = x_max - x_min

                if label in KNOWN_WIDTHS:
                    distance = estimate_distance(KNOWN_WIDTHS[label], box_width)
                    if distance:
                        detected_items.append(f"{label} is {int(distance)} cm away")
                        cv2.rectangle(frame_copy, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                        cv2.putText(frame_copy, f"{label}: {int(distance)} cm", (x_min, y_min - 10),
                                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            # Speak detected items asynchronously
            if detected_items:
                threading.Thread(target=speak_objects, args=(detected_items,), daemon=True).start()

            # Show processed frame
            cv2.imshow("Object Detection", frame_copy)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

# Function to speak detected objects (runs in a separate thread)
def speak_objects(items):
    engine.say(", ".join(items))
    engine.runAndWait()

# Function to continuously listen for voice commands
def listen_for_command():
    global detection_active
    while True:
        with mic as source:
            recognizer.adjust_for_ambient_noise(source)
            try:
                print("Listening for command...")
                audio = recognizer.listen(source, timeout=1)  # Reduced timeout for faster response
                command = recognizer.recognize_google(audio).lower()
                print(f"Command recognized: {command}")

                if command == "start detection":
                    if not detection_active:
                        detection_active = True
                        engine.say("Starting object detection.")
                        engine.runAndWait()
                elif command == "stop detection":
                    if detection_active:
                        detection_active = False
                        engine.say("Stopping detection.")
                        engine.runAndWait()
            except sr.UnknownValueError:
                continue  # Ignore unrecognized audio
            except sr.RequestError:
                print("Speech recognition service unavailable.")

# Function to capture frames efficiently
def capture_frames():
    global frame
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 480)  # Reduce resolution for speed
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 360)
    cap.set(cv2.CAP_PROP_FPS, 30)  # Set higher FPS for smoother capture

    while True:
        ret, new_frame = cap.read()
        if not ret:
            break

        with lock:
            frame = new_frame  # Update global frame

    cap.release()
    cv2.destroyAllWindows()

# Main function
def main():
    # Start the frame capture thread
    frame_thread = threading.Thread(target=capture_frames, daemon=True)
    frame_thread.start()

    # Start the voice command listener thread
    listener_thread = threading.Thread(target=listen_for_command, daemon=True)
    listener_thread.start()

    # Start object detection processing
    detect_objects()

if __name__ == "__main__":
    main()
